# Cumulative Spend Distribution Analysis

This notebook characterizes the underlying cumulative spend-position distribution that feeds the later Beta CDF modeling work. It should be read before `clustered_curve_beta_cdf_model.ipynb`: this page explains the empirical shape of `CUMULATIVEBURNPCT` as a function of `ELAPSEDPCT`, while the later notebook evaluates predictive reference curves against held-out items.

## Dataset and Overall Character

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_lincoln.csv |
| Cluster curve rows | 750 |
| Items | 54 |
| Train rows used for distribution fitting | 625 |
| Train items | 43 |
| Mean cumulative spend pct | 58.80% |
| Median cumulative spend pct | 61.87% |
| Std dev cumulative spend pct | 32.05% |
| Share at completed edge near 100% | 10.67% |
| Share at zero edge near 0% | 0.00% |
| Pearson corr elapsed vs cumulative spend | 0.8049 |
| Spearman corr elapsed vs cumulative spend | 0.8125 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Marginal Distribution of Cumulative Spend

The marginal distribution is bounded between 0 and 1 and is strongly affected by repeated observations from the same item curve. It is not a standalone time-free distribution: a point at 90% elapsed and a point at 20% elapsed should not be expected to have the same cumulative spend behavior. The visible pile-up near 100% is expected because every completed item contributes a final cluster at full cumulative spend.

| Quantile | Cumulative spend pct |
| --- | --- |
| p01 | 0.70% |
| p05 | 5.22% |
| p10 | 9.90% |
| p25 | 31.37% |
| p50 | 61.87% |
| p75 | 88.94% |
| p90 | 100.00% |
| p95 | 100.00% |
| p99 | 100.00% |



## Joint Distribution With Elapsed Percent

The joint view is the core characterization. The distribution is bounded, monotonic in expectation, heteroscedastic, and edge-inflated near completion. The diagonal line is the pure linear spend reference; density above the line is front-loaded spend, while density below it is back-loaded or delayed spend.



## Conditional Distribution by Elapsed Bucket

The table and band plot summarize `CUMULATIVEBURNPCT | ELAPSEDPCT bucket`. The widening and narrowing of the percentile bands matters more than the overall histogram because the production question is where an item sits relative to expected cumulative spend at its current elapsed position.

| Elapsed bucket | Rows | Mean | Median | P10 | P25 | P75 | P90 | IQR | Skew |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0-0.1 | 44 | 10.20% | 6.61% | 0.73% | 1.78% | 16.79% | 21.56% | 15.01% | 1.692 |
| 0.1-0.2 | 69 | 24.32% | 18.86% | 6.31% | 9.60% | 29.05% | 52.90% | 19.44% | 1.716 |
| 0.2-0.3 | 75 | 40.37% | 36.03% | 10.02% | 24.94% | 55.54% | 74.80% | 30.60% | 0.538 |
| 0.3-0.4 | 70 | 51.74% | 50.35% | 26.89% | 39.05% | 64.45% | 87.63% | 25.40% | 0.212 |
| 0.4-0.5 | 64 | 57.20% | 51.33% | 30.59% | 37.98% | 72.89% | 96.63% | 34.92% | 0.266 |
| 0.5-0.6 | 58 | 65.90% | 63.47% | 43.22% | 53.89% | 80.28% | 99.04% | 26.40% | 0.185 |
| 0.6-0.7 | 58 | 77.43% | 74.27% | 59.23% | 65.05% | 91.99% | 98.39% | 26.94% | -0.036 |
| 0.7-0.8 | 46 | 78.68% | 75.88% | 58.52% | 70.85% | 95.44% | 99.77% | 24.60% | -0.122 |
| 0.8-0.9 | 51 | 83.13% | 84.87% | 72.24% | 76.94% | 88.54% | 91.25% | 11.60% | -0.439 |
| 0.9-1.0 | 90 | 97.54% | 100.00% | 92.14% | 96.67% | 100.00% | 100.00% | 3.33% | -2.118 |

## Percentile Bands and Reference Curves

The empirical median curve is generally above the linear reference for much of the item life, which means these clustered payment curves are often front-loaded relative to time. The Beta CDF and anchored polynomial are compact parameterizations of that empirical shape; the later model notebook evaluates the Beta CDF approach more formally.



## Bucketed Density View

This view shows the full shape within each elapsed bucket rather than only percentiles. It makes the conditional spread visible: some elapsed regions are broad and multi-modal because items can receive lumpy clustered payments at different points in their lifecycle.



## Curve Parameterizations

| Model | Alpha / coefficients | Beta | MAE | RMSE | Bias | Clip share | Monotonic violations |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Beta CDF | 0.680026 | 0.984116 | 0.1350 | 0.1834 | 0.0049 |  |  |
| Anchored polynomial degree 3 | 0.987571, 1.037487, -0.964672 |  | 0.1339 | 0.1820 | 0.0016 | 0.0000 | 0 |
| Anchored polynomial degree 4 | 1.000312, 0.759498, 0.335825, -1.419779 |  | 0.1324 | 0.1818 | 0.0002 | 0.0020 | 0 |

The Beta CDF is a useful production candidate because it is naturally bounded and monotone. The anchored polynomial is useful as a flexible descriptive reference, but it is less constrained structurally and should be treated as diagnostic unless monotonicity and stability are explicitly checked.

## Residual Distribution Around Reference Curves

Residuals are `actual cumulative pct - expected cumulative pct`. A good reference curve centers this distribution closer to zero and reduces asymmetric bias. The residual plot shows why the cumulative spend model should not be purely linear: the linear reference leaves a larger systematic position offset where the empirical spend curve is front-loaded.



## Stratification by Duration

Duration buckets have visibly different median cumulative spend curves. This supports the later duration-bucket Beta CDF model: the expected curve is not fully universal across short and long items.



## Stratification by Cluster Count

Cluster count is another proxy for payment cadence and lumpiness. Items with fewer clusters tend to show coarser jumps, while higher-cluster items provide smoother cumulative curves.



## Interpretation

The cumulative spend distribution is best understood as a conditional bounded distribution, not as a single ordinary marginal distribution. Its important properties are:

- Bounded support at `[0, 1]`, with a structural edge at 100% from final clusters.
- Strong positive relationship between elapsed percent and cumulative spend percent.
- A median curve that is not purely linear, indicating systematic front-loading in the clustered payment data.
- Wide conditional dispersion, especially in the middle of the lifecycle, caused by lumpy payment postings and differing payment cadence.
- Meaningful stratification by item duration and cluster count.

This motivates the next-stage notebook: fit expected-position curves and evaluate whether Beta CDF parameterizations outperform a transparent linear reference.